# Zadanie 3: wyszukiwanie semantyczne

Korpus: **wszystkie** wypowiedzi z posiedzeń plenarnych Sejmu X kadencji (2023-11-13 – 2026-10-09, `ppc_cache_sejm-posiedzenia.pkl.gz`), a nie tylko fragmenty anotowane w poprzednich zadaniach.

Modele (tylko polskie / wielojęzyczne, bo korpus jest po polsku):
- bi-enkodery: `sdadas/mmlw-retrieval-roberta-base`, `intfloat/multilingual-e5-base`,
- cross-enkodery: `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (wielojęzyczny MS MARCO), `sdadas/polish-reranker-roberta-v2`,
- LLM (API CLARIN): `bielik-pl-11b-v3.0-Instruct` i `pllum-12b-chat` w RAG,
- pytania testowe do ewaluacji retrievera napisane ręcznie (`queries.json`).

In [1]:
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from openai import OpenAI
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from transformers import AutoModel, AutoTokenizer

SEED = 69
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path("outputs/search")
OUT.mkdir(parents=True, exist_ok=True)
API = "https://services.clarin-pl.eu/api/v1/oapi"
KEY = dict(line.split("=", 1) for line in Path(".env").read_text().split())["CLARIN-PL-API_KEY"]
pd.set_option("display.max_colwidth", 300)

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Korpus

Wypowiedzi z jednej tury (jeden mówca, jedno wystąpienie) sklejamy i dzielimy na równe fragmenty po maks. 250 słów (≈ 512 tokenów, limit modeli).
Odrzucamy komentarze stenografa, 50 najczęstszych formułek proceduralnych i fragmenty krótsze niż 20 słów.

In [2]:
MAX_W, MIN_W = 250, 20

df = pd.read_pickle("ppc_cache_sejm-posiedzenia.pkl.gz")
df = df[(df.who != "komentarz") & (df.text.str.strip() != "")]
df = df[~df.text.isin(df.text.value_counts().head(50).index)]

rows = []
for turn, g in df.groupby("turn", sort=False):
    words = " ".join(g.text).split()
    size = -(-len(words) // -(-len(words) // MAX_W))
    for i in range(0, len(words), size):
        if len(words[i : i + size]) >= MIN_W:
            rows.append((f"{turn}:{i}", " ".join(words[i : i + size]), g.speaker.iloc[0], g.club.iloc[0], str(g.date.iloc[0].date())))

passages = pd.DataFrame(rows, columns=["id", "text", "speaker", "club", "date"])
passages["club"] = passages.club.fillna("bez klubu")
print(len(df), "wypowiedzi ->", len(passages), "fragmentów,", passages.date.min(), "–", passages.date.max())
passages.text.str.split().str.len().describe().round(1)

327019 wypowiedzi -> 91852 fragmentów, 2023-11-13 – 2026-10-09


count    91852.0
mean       154.3
std         64.4
min         20.0
25%        116.0
50%        163.0
75%        208.0
max        250.0
Name: text, dtype: float64

## 1. Tutorial Hugging Face (NLP course, rozdz. 5.6)

Kod z tutoriala (`AutoModel` + CLS pooling, `Dataset.add_faiss_index`, `get_nearest_examples`), z polskim modelem `mmlw-retrieval-roberta-base` zamiast angielskiego.
Model wymaga prefiksu `zapytanie: ` przed zapytaniem.

In [ ]:
tut_ckpt = "sdadas/mmlw-retrieval-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(tut_ckpt)
tut_model = AutoModel.from_pretrained(tut_ckpt).to(DEVICE).eval()


def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]


@torch.no_grad()
def get_embeddings(text_list):
    encoded_input = tokenizer(text_list, padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
    return cls_pooling(tut_model(**encoded_input))


path = OUT / "emb_tutorial.npy"
if path.exists():
    embeddings_dataset = Dataset.from_pandas(passages).add_column("embeddings", list(np.load(path)))
else:
    embeddings_dataset = Dataset.from_pandas(passages).map(
        lambda b: {"embeddings": get_embeddings(b["text"]).float().cpu().numpy()}, batched=True, batch_size=64
    )
    np.save(path, embeddings_dataset.with_format("numpy")["embeddings"])
embeddings_dataset.add_faiss_index(column="embeddings")


def tutorial_search(question, k=5):
    question_embedding = get_embeddings(["zapytanie: " + question]).float().cpu().numpy()
    scores, samples = embeddings_dataset.get_nearest_examples("embeddings", question_embedding, k=k)
    return pd.DataFrame(samples).drop(columns="embeddings").assign(score=scores)[["score", "speaker", "club", "date", "text"]]

Map:   0%|          | 0/91852 [00:00<?, ? examples/s]/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:498.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
100%|██████████| 92/92 [00:00<00:00, 492.92it/s]


In [4]:
np.save(OUT / "emb_tutorial.npy", embeddings_dataset.with_format("numpy")["embeddings"])

TUTORIAL_QUERIES = [
    "Sytuacja na granicy z Białorusią",
    "Czy Tusk to złodziej?",
    "Czy aborcja to morderstwo?",
    "Co Grzegorz Braun mówił o Ukrainie?",
]
for q in TUTORIAL_QUERIES:
    print("\n###", q)
    display(tutorial_search(q))


### Sytuacja na granicy z Białorusią


/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:438.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


,score,speaker,club,date,text
0,69.171539,Maciej Duszczyk,bez klubu,2025-11-20,"granicy polsko-białoruskiej jest dzisiaj spokój, ale to nie znaczy, że spokój będzie za 2 tygodnie. To nic nie znaczy, że Łukaszenka na razie to odwołał i nie przysyła migrantów na granicę polsko-białoruską, sytuacja jest spokojna. Było też pytanie o granicę polsko-białoruską. Proszę państwa, to..."
1,71.729675,Małgorzata Pępek,KO,2026-05-14,"Pani Marszałek! Wysoka Izbo! Bezpieczeństwo granic to fundament spokoju naszych, polskich obywateli. Dlatego przedłużenie ograniczeń na granicy z Białorusią jest konieczne i oczekiwane. Liczby mówią same za siebie. Od stycznia do kwietnia 2025 r. odnotowano ponad 5 tys. prób nielegalnego przekro..."
2,72.569122,Zofia Czernow,KO,2026-03-13,"zagrożeniu. Podejmowane dotąd środki są nadal niewystarczające do jego całkowitego wyeliminowania. Aktualna sytuacja migracyjna na granicy zewnętrznej Rzeczypospolitej Polskiej wskazuje, że w dalszym ciągu ma miejsce presja migracyjna, co powoduje konieczność podjęcia działań w celu zapewnienia ..."
3,72.607086,Zofia Czernow,KO,2026-05-14,"Rzeczypospolitej Polskiej wskazuje, że w dalszym ciągu ma miejsce presja migracyjna, co powoduje konieczność kontynuowania działań w celu zapewnienia stabilizacji sytuacji. Często dochodzi do niebezpiecznych zdarzeń, podczas których agresywni cudzoziemcy atakują polskie patrole, są niszczone ele..."
4,74.010071,Czesław Mroczek,KO,2024-12-19,"Szanowny Panie Marszałku! Szanowni Państwo! Chcę zacząć swoją wypowiedź od przypomnienia danych dotyczących presji migracyjnej, tej nielegalnej migracji na granicy białorusko-polskiej, migracji organizowanej przez służby rosyjskie i białoruskie we współdziałaniu z grupami przestępczymi. W 2022 r..."



### Czy Tusk to złodziej?


,score,speaker,club,date,text
0,108.385132,Władysław Dajczak,PiS,2023-11-28,"Dlatego że wystarczy nie kraść. Tak jak pan minister Czarnek dzisiaj powiedział, nie boimy się tych waszych komisji, jesteśmy spokojni, chyba że one będą prowadzone zgodnie z takim prawem, jak powiedział wasz przewodniczący Donald Tusk: zgodnie z prawem, które będzie takie, jak my je rozumiemy. ..."
1,109.032043,Mariusz Krystian,PiS,2025-04-04,"Panie Marszałku! Wysoka Izbo! Program emerytur trzynastej i czternastej został wprowadzony przez rząd Prawa i Sprawiedliwości, jest to fakt niezaprzeczalny. Niestety w zeszłym roku rząd Donalda Tuska i koalicja 13 grudnia ukradli emerytom część czternastej emerytury. Pierwsze pytanie: Dlaczego u..."
2,111.216949,Bronisław Foltyn,Konfederacja,2026-05-12,"to za bandyta? Proszę państwa, człowiek nazywa się Tomasz Sidorczuk, jest z mojego miasta, z Bielska-Białej, i kupił prezerwatywę na Donalda Tuska. Bandyta nr 1. Nakręcił też filmik w aptece, gdzie kupował tę prezerwatywę za 11 zł. Cała Krajowa Administracja Skarbowa w Katowicach się na niego rz..."
3,112.110352,Dorota Marek,KO,2025-04-02,"Szanowny Panie Marszałku! Pani Minister! Wysoka Izbo! Wydano 366 tys. wiz dla obywateli państw trzecich bez odpowiedniego sprawdzenia, a nawet pod dyktando polityków PiS — takie wnioski płyną z raportu Najwyższej Izby Kontroli dotyczącego afery wizowej. Teraz krzyczycie, że rząd Donalda Tuska ch..."
4,112.432770,Waldemar Andzel,PiS,2026-04-29,"Panie Marszałku! Wysoka Izbo! Za rządów Donalda Tuska albo jesteśmy oszukiwani, albo okradani, albo jeszcze gorsze rzeczy się dzieją. Niech mi ktoś wytłumaczy, czemu zawsze, kiedy rządzi PO lub KO i przystawki, to dochodzi do afer takich jak Amber Gold lub Zondacrypto. Kiedy mieliśmy Amber Gold,..."



### Czy aborcja to morderstwo?


,score,speaker,club,date,text
0,77.085022,Mariusz Krystian,PiS,2024-07-11,"Pani marszałek, Wysoka Izbo, ani jednego więcej, ani jednego nienarodzonego dziecka więcej zabitego tylko dlatego, że lewicowo-liberalne elity oraz grupy polityczne i środowiskowe uznały, że aborcja to prawo, że to kwestia wolności. Aborcja to nie kwestia wolności, to jest zabójstwo człowieka. A..."
1,86.679871,Karina Bosak,Konfederacja,2024-07-11,"Pani Marszałek! Wysoka Izbo! Nieczęsto na tej sali są debaty, w których decyduje się o życiu i śmierci. To jest jedna z nich. Nasz sprzeciw wobec tego projektu ma charakter moralny i ideowy. Zabijanie dzieci nienarodzonych to zło w czystej postaci. Tworzenie warunków, w których kobiety można do ..."
2,91.494568,Paweł Szefernaker,PiS,2024-07-11,"Pani marszałek, Wysoka Izbo, naród, który zabija swoje dzieci, jest narodem bez przyszłości. To słowa wielkiego Polaka, św. Jana Pawła II. Bezkarność aborcji wprowadzali na terenie naszego kraju ci, którzy nam jak najgorzej życzyli, którzy nie życzyli dobrze Rzeczypospolitej. Aborcja nie jest pr..."
3,92.099213,Karina Bosak,Konfederacja,2024-11-06,"się od poczęcia, od tego momentu człowiek ma niezbywalne prawo do życia i należy go chronić. Przestańcie kłamać, że projekt dotyczy tylko bezkarności osób nakłaniających do zabijania człowieka, kiedy jest jeszcze w łonie matki, czy w tym pomagających. Chcecie nim wykreślić przestępstwa, za które..."
4,92.464233,Roman Fritz,Konfederacja,2024-04-11,"Grupa feministek pomagająca mordować nienarodzone dzieci, Aborcyjny Dream Team, pochwaliła się mniej więcej 2 lata temu, że pomogła zabić dziecko gotowe do porodu. Jego matka była w 37. tygodniu ciąży. Feministki przyznają, że pomaganie w aborcjach jest ich życiową pasją i misją, a za cel obrały..."



### Co Grzegorz Braun mówił o Ukrainie?


,score,speaker,club,date,text
0,88.170898,Dorota Olko,Lewica,2025-02-05,"matki II RP mogli ostro spierać się nawet o kształt ustrojowy państwa, ale w kwestii granic, w kwestii walki z sowieckim zagrożeniem zachowywali daleko idącą jednomyślność. Pokaz podobnej politycznej jednomyślności mieliśmy w Polsce całkiem niedawno, bo po 24 lutego 2022 r. Jestem i byłam dumna,..."
1,92.510345,Barbara Grygorcewicz,KO,2025-12-19,"Szanowny Panie Marszałku! Wysoka Izbo! W udzielonym ostatnio wywiadzie Karol Nawrocki stwierdził, jakoby Donald Tusk miał szkodzić Polsce bardziej niż Grzegorz Braun. Słowa takie są dla mnie nie tylko niezrozumiałe, lecz także absurdalne. Donald Tusk jest premierem rządu, który po 8 latach chaos..."
2,93.921478,Paweł Śliz,Polska2050-TD,2025-07-25,"Panie Marszałku! Wysoka Izbo! Dawno nie miałem takich emocji, mówiąc z tego miejsca, bo to, co usłyszeliśmy z ust Brauna, zasługuje na potępienie. To jest skandal, że musimy debatować i stwierdzać coś, co jest oczywiste. Ja nie tylko powiem, że Braun jest winny. Ale on powinien ponieść surową ka..."
3,94.073914,Michał Szczerba,KO,2024-04-26,"zobaczcie państwo — po raz kolejny służyła przedstawicielom skrajnej, nacjonalistycznej, brunatnej prawicy do tego, żeby uderzyć w wartości, jakie prezentuje sobą Unia Europejska, w czasie, kiedy trwa pełnoskalowa wojna na Ukrainie, wywołana przez zbrodniczy reżim Władimira Putina, panie pośle B..."
4,94.077972,Joanna Lichocka,PiS,2025-07-25,"Panie Marszałku! Wysoki Sejmie! Prawo i Sprawiedliwość popiera projekt tej uchwały. Pretekstem do jej powstania była bardzo zła, skandaliczna wypowiedź Grzegorza Brauna, którą powszechnie zinterpretowano jako negowanie niemieckich zbrodni w obozach koncentracyjnych i Holokaustu. My się na to nie..."


### Analiza wyników tutoriala

`add_faiss_index` domyślnie używa odległości L2 na nieznormalizowanych osadzeniach CLS, więc `score` to odległość (mniej = bliżej), a nie podobieństwo.

- **Granica z Białorusią** – wszystkie 5 fragmentów trafia w temat (presja migracyjna, przedłużanie ograniczeń, Łukaszenka). Zapytanie neutralne, opisowe, z wieloma pasującymi wypowiedziami – najlepszy przypadek.
- **„Czy Tusk to złodziej?”** – wyszukiwarka nie odpowiada na pytanie, tylko znajduje wypowiedzi z tym samym słownictwem: „wystarczy nie kraść”, „rząd Donalda Tuska ukradli emerytom część czternastej emerytury”, „za rządów Donalda Tuska albo jesteśmy oszukiwani, albo okradani”. Trafia też fragment o „bandycie nr 1”, który kupił prezerwatywę na Tuska, i jedna wypowiedź KO odbijająca zarzut (afera wizowa). Odległości (108–112) są o ok. 40 większe niż dla granicy – krótkie, prowokacyjne pytanie leży daleko od długich przemówień.
- **„Czy aborcja to morderstwo?”** – wszystkie 5 wyników to PiS i Konfederacja („to jest zabójstwo człowieka”, „naród, który zabija swoje dzieci”). Słowo „morderstwo” jest bliskie językowi tylko jednej strony debaty, a zwolennicy liberalizacji mówią o „prawie” i „wolności”. Sformułowanie zapytania przesądza, czyje głosy zobaczymy.
- **„Co Grzegorz Braun mówił o Ukrainie?”** – błąd wyszukiwania. Zwracane są wypowiedzi innych posłów *o* Braunie (potępienie słów o Holokauście, porównanie Tusk–Braun przez Nawrockiego), a nie jego własne słowa o Ukrainie. Mówca jest metadaną, a nie częścią tekstu, więc osadzenie go nie widzi; model łapie nazwisko i „Ukrainę” osobno.

Wnioski: wyszukiwanie semantyczne dobrze radzi sobie z pytaniami o temat, ale zwraca to, co *podobne* do zapytania, a nie to, co *prawdziwe* – na pytania wartościujące odpowiada cytatami jednej strony. Pytania o konkretnego mówcę wymagają filtrowania po metadanych.

## 2. Retriever (bi-enkoder)

### Pytania testowe

Posiedzenia nie mają par pytanie→dokument (jak `question`/`answer` w SQuAD), więc przygotowaliśmy je ręcznie (`queries.json`):
z 200 losowych fragmentów wypowiedzi posłów (seed 69) do każdego merytorycznego fragmentu napisano jedno pytanie, na które on odpowiada,
bez dosłownego cytowania. 35 fragmentów czysto proceduralnych (zapraszanie mówców, powitania gości) pominięto. Relewantny jest dokładnie fragment źródłowy.

In [5]:
position = pd.Series(passages.index, index=passages.id)
queries = pd.read_json("queries.json").assign(gold=lambda d: position[d.id].values)
print(len(queries), "pytań")
queries.sample(8, random_state=SEED).assign(fragment=lambda d: passages.text[d.gold].str[:250].values)[["question", "fragment"]]

165 pytań


,question,fragment
20,Ile pieniędzy z Rządowego Funduszu Rozwoju Dróg trafiło do gmin i powiatów?,Szanowny Panie Marszałku! Panie Ministrze! Rządowy Fundusz Rozwoju Dróg jest jednym z kluczowych instrumentów wsparcia finansowego dla jednostek samorządu terytorialnego w Polsce w zakresie rozwoju infrastruktury drogowej. Jego głównym celem jest pop
154,Po co PiS proponuje ustawę o rzeczniku praw rolników?,"krytykujecie naszą ustawę o rzeczniku rolników. My dbamy o polskich rolników i dlatego taka ustawa jest proponowana. Dziękuję posłowi Andrzejowi Kryjowi. Jest Andrzej Kryj. Bardzo dobre wystąpienie, które mówi o tym, że właśnie taka ustawa jest potrz"
94,Jaki bonus finansowy otrzyma metropolia pomorska w latach 2027–2029?,"żaden sposób nie wpłynie to na zmianę modelu funkcjonowania samorządności — to jesteśmy otwarci, żeby te tematy przedyskutować. Pula środków. Szanowni państwo, faktycznie mamy mechanizm korygujący na 2 lata. Pomiędzy 2027 r. a 2029 r. metropolia pomo"
128,Czy urzędy stanu cywilnego mogą odmawiać transkrypcji zagranicznych aktów małżeństw jednopłciowych?,"Szanowny Panie Marszałku! Wysoka Izbo! 22 maja podpisane zostało rozporządzenie ministra cyfryzacji oraz ministra spraw wewnętrznych i administracji wprowadzające nowe wzory aktów małżeństwa, umożliwiające transkrypcję zagranicznych aktów małżeństw j"
79,Jak pamiętać o rzezi wołyńskiej i jednocześnie budować pojednanie z Ukraińcami?,"Pani Marszałkini! Wysoki Sejmie! Pamięć nosi się w sercu, a nie na klapie marynarki. Dla mnie ta uchwała ma także pewien wymiar osobisty. Mój dziadek spod przemyskiej wsi Grochowce przeżył wraz z rodziną, ponieważ pewnego dnia Ukrainiec, jego sąsiad"
36,Jakie ryzyko dla mieszkańców wsi niesie przeniesienie kompetencji sądów na notariuszy?,"podkreślić, że wszelkie rozwiązania deregulacyjne mają pomagać, a nie wykluczać czy pogłębiać nierówności. Pamiętajmy o czasami ograniczonym dostępie do notariusza w małych miejscowościach. Pamiętajmy przy przenoszeniu części kompetencji sądów na not"
8,Czy w postępowaniu upadłościowym można ujawniać informacje objęte tajemnicą bankową?,"udzielenia wspomnianych wcześniej informacji. Brak podstawy prawnej powoduje liczne wątpliwości interpretacyjne i problemy w praktyce, tym bardziej że ujawnienie lub wykorzystanie informacji stanowiących tajemnicę bankową lub zawodową niezgodnie z up"
103,Jak państwo chroni obywateli przed dezinformacją w mediach?,"dezinformacji. Co zrobić, żeby wśród odbiorców mediów — a przecież wiemy, że media dzisiaj są naprawdę ogólnie dostępne, tak powszechne, że pewnie połowa społeczeństwa albo większa jego część nie wyobraża sobie życia bez dostępu do mediów — ta dezinf"


### Metryki

Na każde pytanie jest dokładnie jeden relewantny fragment, więc (wg artykułu Pinecone o ewaluacji offline):
- **Recall@k** – czy relewantny fragment jest w top-k,
- **MRR@k** – średnia z 1/pozycja relewantnego fragmentu (0 gdy poza top-k),
- **nDCG@k** – przy jednym relewantnym dokumencie: 1/log2(pozycja + 1).

In [6]:
def evaluate(ranked, gold, k=5):
    hit = np.asarray(ranked)[:, :k] == np.asarray(gold)[:, None]
    rank = np.where(hit.any(1), hit.argmax(1) + 1, np.inf)
    return {"Recall@1": (rank == 1).mean(), f"Recall@{k}": (rank <= k).mean(), f"MRR@{k}": (1 / rank).mean(), f"nDCG@{k}": (1 / np.log2(rank + 1)).mean()}


evaluate(np.c_[queries.gold, np.zeros((len(queries), 4), int)], queries.gold)

{'Recall@1': np.float64(1.0),
 'Recall@5': np.float64(1.0),
 'MRR@5': np.float64(1.0),
 'nDCG@5': np.float64(1.0)}

### Osadzenia (sentence-transformers)

In [7]:
MODELS = {
    "mmlw-roberta-base": ("sdadas/mmlw-retrieval-roberta-base", "zapytanie: ", ""),
    "multilingual-e5-base": ("intfloat/multilingual-e5-base", "query: ", "passage: "),
}
encoders, corpus_emb, query_emb = {}, {}, {}
for key, (name, q_prefix, d_prefix) in MODELS.items():
    encoders[key] = SentenceTransformer(name, device=DEVICE)
    path = OUT / f"emb_{key}.npy"
    if not path.exists():
        np.save(path, encoders[key].encode([d_prefix + t for t in passages.text], batch_size=64, normalize_embeddings=True, show_progress_bar=True))
    corpus_emb[key] = np.load(path)
    query_emb[key] = encoders[key].encode([q_prefix + q for q in queries.question], normalize_embeddings=True)
    print(key, corpus_emb[key].shape)

Batches: 100%|██████████| 1436/1436 [05:06<00:00,  4.68it/s]


mmlw-roberta-base (91852, 768)


Batches: 100%|██████████| 1436/1436 [14:56<00:00,  1.60it/s]


multilingual-e5-base (91852, 768)


### Wyszukiwanie bez indeksu (`util.semantic_search`, top_k = 5)

In [8]:
def brute_force(C, Q, k=5):
    hits = util.semantic_search(torch.from_numpy(Q), torch.from_numpy(C), top_k=k)
    return np.array([[h["corpus_id"] for h in hs] for hs in hits])


def semantic_search(question, key="mmlw-roberta-base", k=5):
    q = encoders[key].encode([MODELS[key][1] + question], normalize_embeddings=True)
    hits = util.semantic_search(torch.from_numpy(q), torch.from_numpy(corpus_emb[key]), top_k=k)[0]
    return passages.iloc[[h["corpus_id"] for h in hits]].assign(score=[h["score"] for h in hits])[["score", "speaker", "club", "date", "text"]]


for i in [0, 1, 2]:
    q = queries.iloc[i]
    print("\n###", q.question, "| relewantny:", q.gold)
    for key in MODELS:
        print(key)
        display(semantic_search(q.question, key))


### Dlaczego Lewica chciała, żeby w zespole wybierającym dyrektora CMKP była przedstawicielka pielęgniarek? | relewantny: 24896
mmlw-roberta-base


,score,speaker,club,date,text
24932,0.826840,Urszula Demkow,bez klubu,2024-11-08,"medycznych. Tego będzie dotyczyć część zapisów statutowych, które zapewnią wszystkim zawodom medycznym, nie tylko pielęgniarkom i lekarzom, ale także diagnostom, fizjoterapeutom, ratownikom medycznym, farmaceutom i innym możliwość decydowania o kształceniu danej grupy zawodowej. Myślę, że najlep..."
24896,0.825281,Joanna Wicha,Lewica,2024-11-08,w życie ustawy o zawodzie ratownika medycznego. Umożliwia również powrót do zawodu po przerwie w jego wykonywaniu. Wprowadza przepisy regulujące zatwierdzenie zrealizowanego obowiązku doskonalenia zawodowego. Największe obawy środowiska ratowników medycznych do czasu powołania samorządu zawodowe...
24925,0.815501,Elżbieta Gelert,KO,2024-11-08,"Pani Marszałek! Pani Minister! Wysoka Izbo! Właściwie poruszamy cały czas temat wyboru dyrektora CMKP, natomiast zabrakło w dyskusji nawet na posiedzeniu Komisji Zdrowia kwestii dotyczącej tego, jak naprawdę funkcjonowało CMKP do tej pory. Mówicie państwo, że tak fantastycznie, może nie fantasty..."
24881,0.814508,Krzysztof Bojarski,KO,2024-11-08,Rozwiązania zawarte w projekcie umożliwiają rozwój zawodowy pielęgniarkom i położnym oraz ułatwiają dostęp do prowadzenia działalności naukowej i dydaktycznej. Projekt wprowadza zmiany w zakresie działalności CMKP wynikające z przeniesienia zadań w zakresie kształcenia podyplomowego pielęgniarek...
24890,0.811202,Wioleta Tomczak,Polska2050-TD,2024-11-08,"członków połączonych komisji, przez co dziś w projekcie nowelizacji ustawy możemy podkreślić gwarancję wyboru na stanowisko dyrektora CMKP osoby, która posiada wiedzę, kompetencje i wykonuje zawód medyczny, co ściśle związane jest z potrzebami wynikającymi z charakteru zadań CMKP. Proponowany pr..."


multilingual-e5-base


,score,speaker,club,date,text
24896,0.883829,Joanna Wicha,Lewica,2024-11-08,w życie ustawy o zawodzie ratownika medycznego. Umożliwia również powrót do zawodu po przerwie w jego wykonywaniu. Wprowadza przepisy regulujące zatwierdzenie zrealizowanego obowiązku doskonalenia zawodowego. Największe obawy środowiska ratowników medycznych do czasu powołania samorządu zawodowe...
24929,0.869225,Urszula Demkow,bez klubu,2024-11-08,"Pani Marszałek! Wysoka Izbo! Dziękuję bardzo za państwa głosy, zarówno te popierające, jak i te krytyczne. Chciałabym jakoś w uporządkowany sposób odpowiedzieć na państwa pytania. Pierwsze pytanie, pytanie, jakie wysuwa mi się na pierwszy plan, brzmi: Czemu ma służyć połączenie tych dwóch instyt..."
24915,0.864004,Czesław Hoc,PiS,2024-11-08,"Szanowna Pani Marszałek! Państwo Ministrowie! Czy nie razi was kreowanie w tej ustawie wyraźnego konfliktu interesów? Oto powołujecie zespół konkursowy, który będzie składał się z trzech przedstawicieli Naczelnej Rady Lekarskiej i trzech przedstawicieli Ministerstwa Zdrowia. Oczywiście teraz pop..."
24918,0.863267,Iwona Kozłowska,KO,2024-11-08,Szanowna Pani Marszałek! Pani Minister! Wysoka Izbo! Jedną ze zmian w projekcie ustawy jest przejęcie przez Centrum Medyczne Kształcenia Podyplomowego zadań Centrum Kształcenia Podyplomowego Pielęgniarek i Położnych. Mam pytanie: Z jakiego powodu Ministerstwo Zdrowia podjęło decyzję o włączeniu ...
24894,0.862406,Joanna Wicha,Lewica,2024-11-08,"Panie Marszałku! Wysoka Izbo! Mam nadzieję, że skonsolidowana, wzmocniona jednostka kształcenia będzie lepiej odpowiadać na potrzeby pacjentów i pracowników polskiego systemu ochrony zdrowia. Mam też nadzieję, że jest to krok w stronę rozwiązania problemu braku specjalistów w wielu dziedzinach, ..."



### Jakie finansowe skutki dla Polski miał niekonstytucyjny wybór sędziów do KRS przez Sejm? | relewantny: 9223
mmlw-roberta-base


,score,speaker,club,date,text
68277,0.839546,Tomasz Zimoch,Polska2050-TD,2026-02-26,"przepisów ustawy o KRS-ie za niezgodne z konstytucją. Jako zabezpieczenie sędziowie ci, członkowie KRS-u, zażądali zakazu działania Sejmu — zakazu czynności prawnych i faktycznych, jeśli chodzi o wybory do KRS. To jest prawda. To nie jest bajka. Uchwała Sejmu ma charakter deklaratoryjny, informa..."
91361,0.838559,Aleksandra Leo,Polska2050-TD,2026-10-08,"Panie Marszałku! Wysoka Izbo! To, co Prawo i Sprawiedliwość zrobiło z Krajową Radą Sądownictwa i polskim sądownictwem, to skandal. Pozbawił KRS niezależności i oddał prawicowym politykom wpływ na wybór sędziów. Skutki tych decyzji widać dziś w sądach. Strony kwestionują prawidłowość składów, a o..."
18899,0.835889,Michał Pyrzyk,PSL-TD,2024-09-12,"prewencyjnej do Trybunału Konstytucyjnego, czym tak de facto zablokował wejście w życie tej dobrej nowelizacji. Należy zatem przypomnieć, że niewłaściwie ukształtowana KRS wprowadziła niesamowity chaos w polskim sądownictwie. Wszystkie nominacje sędziowskie dokonane na wniosek tak ukształtowanej..."
6723,0.835861,Witold Tumanowicz,Konfederacja,2024-02-21,"Panie Marszałku! Wysoka Izbo! Koalicja uśmiechniętej Polski przejęła rządy i od razu po powołaniu do podobno niekonstytucyjnej KRS posłowie koalicji pobrali ogromne wynagrodzenia — te same wynagrodzenia, za których pobieranie krytykowali nominatów PiS. Tylko w grudniu 2023 r. poseł Kropiwnicki —..."
9171,0.831132,Sebastian Kaleta,PiS,2024-04-10,"wybór w wyborach powszechnych. Co więcej, z uwagi na być może negatywne skutki takich konkretnych rozwiązań ustrojodawca dał elastyczność co do zmiany takiego prawa i na tym właśnie polega spójność tej koncepcji. A więc nie sposób powiedzieć, że ustawa jest niekonstytucyjna. Ale co wy, państwo, ..."


... (2 outputs left out)


### Jakie zastrzeżenia zgłaszano do zakazu pełnienia funkcji asystenta osobistego przez członków rodziny osoby niesamodzielnej? | relewantny: 85614
mmlw-roberta-base


,score,speaker,club,date,text
85614,0.827267,Dariusz Matecki,PiS,2026-09-02,"Dlaczego w art. 87 projektu wprowadzono bezwzględny zakaz pełnienia funkcji asystenta osobistego przez najbliższych członków rodziny (małżonków, dzieci, rodziców, rodzeństwo, powinowatych oraz osoby wspólnie gospodarujące), co uderza w podstawową komórkę społeczną, pozbawia osoby niesamodzielne ..."
85547,0.811634,Joanna Frydrych,KO,2026-09-02,"Dlatego asystencja osobista musi stać się stabilnym, przewidywalnym elementem systemu wsparcia. I właśnie to robimy. Ale chcę powiedzieć jeszcze jedną rzecz. Asystent osobisty to nie ktoś, kto ma decydować za osobę z niepełnosprawnością. Wręcz przeciwnie — asystent ma pomóc jej decydować o własn..."
21186,0.806215,Katarzyna Ueberhan,Lewica,2024-10-01,"osobistej dla dzieci i młodzieży, brak możliwości swobodnego wyboru realizatora asystencji osobistej, brak możliwości samodzielnego zatrudnienia asystenta osobistego przez użytkownika, brak możliwości wykonania czynności pielęgnacyjno-medycznych przez asystenta osobistego po przeszkoleniu i przy..."
21214,0.806131,Małgorzata Niemczyk,KO,2024-10-01,"Panie Marszałku! Wysoka Izbo! Asystent osobisty wspiera osobę z niepełnosprawnością w codziennych czynnościach, których ona sama nie może wykonać. Dla całej rzeszy osób z niepełnosprawnością takie wsparcie to warunek konieczny do prowadzenia niezależnego życia i też prawo tych osób, które wynika..."
85545,0.806030,Maja Nowak,Polska2050-TD,2026-09-02,"ustawowa asystencja osobista. Możemy dyskutować o tym, czy system powinien od początku obejmować więcej osób, możemy spierać się o progi punktowe, liczbę godzin czy adekwatne finansowanie. Możemy, i zapewne będziemy, wracać do części tych kwestii także po uruchomieniu tego systemu i po zebraniu ..."


multilingual-e5-base


,score,speaker,club,date,text
85614,0.854531,Dariusz Matecki,PiS,2026-09-02,"Dlaczego w art. 87 projektu wprowadzono bezwzględny zakaz pełnienia funkcji asystenta osobistego przez najbliższych członków rodziny (małżonków, dzieci, rodziców, rodzeństwo, powinowatych oraz osoby wspólnie gospodarujące), co uderza w podstawową komórkę społeczną, pozbawia osoby niesamodzielne ..."
39285,0.842394,Maja Nowak,Polska2050-TD,2025-05-08,"ukończeniem 65. roku życia, mogły kontynuować tę formę wsparcia również po przekroczeniu tego wieku? To rozwiązanie spotkało się z bardzo dużym poparciem w trakcie konsultacji społecznych i byłoby wyrazem troski o bezpieczeństwo oraz ciągłość i stabilność świadczonego wsparcia. I trzecie pytanie..."
21214,0.840568,Małgorzata Niemczyk,KO,2024-10-01,"Panie Marszałku! Wysoka Izbo! Asystent osobisty wspiera osobę z niepełnosprawnością w codziennych czynnościach, których ona sama nie może wykonać. Dla całej rzeszy osób z niepełnosprawnością takie wsparcie to warunek konieczny do prowadzenia niezależnego życia i też prawo tych osób, które wynika..."
21217,0.839042,Małgorzata Pępek,KO,2024-10-01,"Panie Marszałku! Wysoka Izbo! Uwagi do projektu przedstawił m.in. Urząd Ochrony Danych Osobowych, zwracając uwagę na fakt, że świadczenie usług asystencji osobistej wymaga przetwarzania danych zarówno osób z niepełnosprawnościami i ich rodzin, jak i samych asystentów. Chodzi tutaj nie tylko o da..."
85547,0.838516,Joanna Frydrych,KO,2026-09-02,"Dlatego asystencja osobista musi stać się stabilnym, przewidywalnym elementem systemu wsparcia. I właśnie to robimy. Ale chcę powiedzieć jeszcze jedną rzecz. Asystent osobisty to nie ktoś, kto ma decydować za osobę z niepełnosprawnością. Wręcz przeciwnie — asystent ma pomóc jej decydować o własn..."


### Indeks FAISS

Wg guideline'u FAISS dla ~80 tys. wektorów: `IndexFlatIP` (dokładny, iloczyn skalarny = kosinus dla znormalizowanych wektorów),
`IndexHNSWFlat` (graf, M = 32, efSearch = 64) i `IndexIVFFlat` (1024 ≈ 4·√N centroidów, nprobe = 16).

In [9]:
def build_index(kind, C):
    d = C.shape[1]
    if kind == "Flat":
        index = faiss.IndexFlatIP(d)
    elif kind == "HNSW":
        index = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
        index.hnsw.efSearch = 64
    else:
        index = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, 1024, faiss.METRIC_INNER_PRODUCT)
        index.train(C)
        index.nprobe = 16
    index.add(C)
    return index


results, indexes, ranked = [], {}, {}
for key in MODELS:
    C, Q = corpus_emb[key], query_emb[key]
    t = time.perf_counter()
    ranked[key, "bez indeksu"] = brute_force(C, Q)
    results.append({"model": key, "metoda": "bez indeksu", "budowa [s]": 0.0, "zapytanie [ms]": (time.perf_counter() - t) / len(Q) * 1000})
    for kind in ["Flat", "HNSW", "IVF"]:
        t = time.perf_counter()
        indexes[key, kind] = build_index(kind, C)
        build = time.perf_counter() - t
        t = time.perf_counter()
        ranked[key, kind] = indexes[key, kind].search(Q, 5)[1]
        results.append({"model": key, "metoda": f"FAISS {kind}", "budowa [s]": build, "zapytanie [ms]": (time.perf_counter() - t) / len(Q) * 1000})

comparison = pd.DataFrame([
    r | evaluate(ranked[r["model"], r["metoda"].removeprefix("FAISS ")], queries.gold)
    | {"zgodność top-5 z bez indeksu": np.mean([len(set(a) & set(b)) / 5 for a, b in zip(ranked[r["model"], r["metoda"].removeprefix("FAISS ")], ranked[r["model"], "bez indeksu"])])}
    for r in results
])
comparison.round(3)

,model,metoda,budowa [s],zapytanie [ms],Recall@1,Recall@5,MRR@5,nDCG@5,zgodność top-5 z bez indeksu
0,mmlw-roberta-base,bez indeksu,0.000,1.024,0.297,0.552,0.389,0.430,1.000
1,mmlw-roberta-base,FAISS Flat,0.156,1.377,0.303,0.552,0.392,0.432,1.000
2,mmlw-roberta-base,FAISS HNSW,4.732,0.099,0.303,0.545,0.391,0.429,0.981
3,mmlw-roberta-base,FAISS IVF,2.551,0.079,0.273,0.485,0.349,0.383,0.915
4,multilingual-e5-base,bez indeksu,0.000,0.753,0.370,0.570,0.439,0.471,1.000
5,multilingual-e5-base,FAISS Flat,0.039,1.231,0.370,0.570,0.439,0.471,1.000
6,multilingual-e5-base,FAISS HNSW,4.961,0.110,0.370,0.570,0.438,0.471,0.979
7,multilingual-e5-base,FAISS IVF,2.395,0.120,0.309,0.515,0.382,0.415,0.844


### Analiza retrievera

- **Modele**: `multilingual-e5-base` jest lepszy od `mmlw-roberta-base` (Recall@5 0,570 vs 0,552, MRR@5 0,439 vs 0,389), mimo że mmlw był trenowany na polskim. E5 wyraźniej częściej stawia relewantny fragment na 1. miejscu (Recall@1 0,370 vs 0,297).
- **Bez indeksu vs FAISS Flat**: wyniki identyczne (zgodność top-5 = 1,0; minimalne różnice w mmlw to remisy w precyzji float). Flat to ten sam przegląd zupełny co `util.semantic_search`, więc ani nie zyskuje, ani nie traci.
- **HNSW**: ok. 10× szybsze zapytanie (0,1 ms vs 1,2 ms), 98% zgodności top-5 i praktycznie bez straty jakości. Koszt to ~5 s budowy indeksu.
- **IVF** (nprobe = 16 z 1024 list, czyli przeszukujemy ~1,6% korpusu): najszybsze, ale traci 5–7 p.p. Recall@5. Relewantny fragment często leży w nieprzeszukanej komórce.
- Przy ~92 tys. fragmentów przegląd zupełny jest jeszcze tani. HNSW opłaca się dopiero przy dużo większej liczbie zapytań lub dokumentów, a IVF wymagałby większego nprobe.

**Dlaczego Recall@5 to tylko ~0,57?** W korpusie jest wiele prawie identycznych fragmentów z jednej debaty. W pytaniu o przedstawicielkę pielęgniarek w CMKP wszystkie 5 wyników pochodzi z tej samej debaty z 2024-11-08. W pytaniu o KRS e5 na 1. miejscu stawia inne przemówienie tej samej posłanki (Elżbieta Polak) z miesiąca wcześniej. Ocena uznaje za relewantny tylko jeden fragment źródłowy, więc wiele „pomyłek” to w rzeczywistości poprawne odpowiedzi (fałszywe negatywy w ewaluacji). Zmierzone metryki są więc dolnym ograniczeniem jakości.

## 3. Re-ranker (cross-enkoder)

Lepszy bi-enkoder z indeksem FAISS Flat zwraca 50 kandydatów, cross-enkoder ocenia pary (pytanie, fragment) i zwraca top_k = 5.

In [10]:
best = comparison[comparison.metoda == "FAISS Flat"].sort_values("MRR@5").model.iloc[-1]
N_CAND = 50
candidates = indexes[best, "Flat"].search(query_emb[best], N_CAND)[1]
print(best, {f"Recall@{N_CAND}": evaluate(candidates, queries.gold, N_CAND)[f"Recall@{N_CAND}"]})

RERANKERS = {
    "mmarco-mMiniLMv2-L12": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "polish-reranker-roberta-v2": "sdadas/polish-reranker-roberta-v2",
}


def rerank(ce, questions, cand, k=5):
    pairs = [(q, passages.text.iat[i]) for q, row in zip(questions, cand) for i in row]
    scores = ce.predict(pairs, batch_size=32, show_progress_bar=True).reshape(cand.shape)
    return np.take_along_axis(cand, np.argsort(-scores, 1), 1)[:, :k]


cross_encoders, rerank_rows = {}, [{"potok": f"{best} (bez rerankingu)", "czas [ms/zapytanie]": None} | evaluate(candidates, queries.gold)]
reranked = {}
for key, name in RERANKERS.items():
    cross_encoders[key] = CrossEncoder(name, device=DEVICE)
    t = time.perf_counter()
    reranked[key] = rerank(cross_encoders[key], queries.question, candidates)
    rerank_rows.append({"potok": f"{best} + {key}", "czas [ms/zapytanie]": (time.perf_counter() - t) / len(queries) * 1000} | evaluate(reranked[key], queries.gold))
pd.DataFrame(rerank_rows).round(3)

multilingual-e5-base {'Recall@50': np.float64(0.8606060606060606)}


Batches: 100%|██████████| 258/258 [01:20<00:00,  3.19it/s]


,potok,czas [ms/zapytanie],Recall@1,Recall@5,MRR@5,nDCG@5
0,multilingual-e5-base (bez rerankingu),NaN,0.370,0.570,0.439,0.471
1,multilingual-e5-base + mmarco-mMiniLMv2-L12,222.310,0.455,0.661,0.536,0.567
2,multilingual-e5-base + polish-reranker-roberta-v2,491.161,0.503,0.739,0.596,0.632


In [11]:
def gold_rank(rows):
    hit = np.asarray(rows) == queries.gold.to_numpy()[:, None]
    return np.where(hit.any(1), hit.argmax(1) + 1, 0)


best_ce = max(RERANKERS, key=lambda k: evaluate(reranked[k], queries.gold)["MRR@5"])
changes = queries[["question"]].assign(przed=gold_rank(candidates[:, :5]), po=gold_rank(reranked[best_ce]))
print(best_ce, "| pozycja relewantnego fragmentu (0 = poza top-5):")
display(pd.crosstab(changes.przed, changes.po))
for label, d in [("poprawa", changes[(changes.po > 0) & ((changes.przed == 0) | (changes.po < changes.przed))]),
                 ("pogorszenie", changes[(changes.przed > 0) & ((changes.po == 0) | (changes.po > changes.przed))])]:
    print("\n##", label, len(d))
    for i, r in d.head(2).iterrows():
        print("\n", r.question, f"(przed: {r.przed}, po: {r.po})")
        print("  relewantny:", passages.text.iat[queries.gold[i]][:300])
        print("  top-1 po rerankingu:", passages.text.iat[reranked[best_ce][i, 0]][:300])

polish-reranker-roberta-v2 | pozycja relewantnego fragmentu (0 = poza top-5):


po,0,1,2,3,4,5
przed,,,,,,
0,39,16,8,2,4,2
1,2,47,6,3,2,1
2,0,7,1,1,0,1
3,0,8,1,1,0,0
4,2,3,3,0,0,1
5,0,2,2,0,0,0



## poprawa 58

 Jakie finansowe skutki dla Polski miał niekonstytucyjny wybór sędziów do KRS przez Sejm? (przed: 2, po: 1)
  relewantny: Panie Marszałku! Panowie Ministrowie! Wysoka Izbo! Tylko niezależny sąd: konstytucyjny, administracyjny, karny, cywilny jest w stanie zapewnić obywatelom sprawiedliwy wyrok. Niezgodne z konstytucją przeniesienie kompetencji wyboru 15 sędziów do KRS-u na Sejm niesie konsekwencje nie tylko prawne, ale
  top-1 po rerankingu: Panie Marszałku! Panowie Ministrowie! Wysoka Izbo! Tylko niezależny sąd: konstytucyjny, administracyjny, karny, cywilny jest w stanie zapewnić obywatelom sprawiedliwy wyrok. Niezgodne z konstytucją przeniesienie kompetencji wyboru 15 sędziów do KRS-u na Sejm niesie konsekwencje nie tylko prawne, ale

 Czy w wydatkach na obronność uwzględniono emerytury wojskowe ze starego portfela? (przed: 0, po: 5)
  relewantny: Na koniec chciałbym powiedzieć, że w tym mieści się również portfel emerytów żołnierzy, tzw. stary portfel. Wszystkich so

### Analiza re-rankera

| potok | Recall@1 | Recall@5 | MRR@5 | ms/zapytanie |
|---|---|---|---|---|
| e5 + FAISS | 0,370 | 0,570 | 0,439 | ~1 |
| + mmarco-mMiniLMv2 | 0,455 | 0,661 | 0,536 | 222 |
| + polish-reranker-roberta-v2 | **0,503** | **0,739** | **0,596** | 491 |

- Re-ranking to największy zysk w całym potoku: polski cross-enkoder podnosi Recall@5 o 17 p.p., a MRR@5 o 16 p.p. Wielojęzyczny mmarco daje mniej więcej połowę tego zysku przy połowie czasu.
- Ceną jest czas: cross-enkoder ocenia 50 par na zapytanie, więc wychodzi kilkaset razy wolniej niż sam bi-enkoder. Dlatego stosuje się go tylko do kandydatów.
- Sufit wyznacza retriever. Recall@50 = 0,86, więc dla 14% pytań relewantnego fragmentu nie ma wśród kandydatów i re-ranker nic nie zmieni.
- Tabela przejść: 32 pytania wyciągnięte spoza top-5 (16 od razu na 1. miejsce), 47 zostaje na 1. miejscu, 20 awansuje na 1. z pozycji 2–5. Tylko 4 wypadają z top-5. Bilans to 58 popraw wobec 19 pogorszeń.
- Pogorszenia to głównie artefakt ewaluacji. W pytaniu o promocję 800+ re-ranker wybiera fragment z konkretną kwotą („na samą promocję pikników wydano 1,4 mln zł”), który odpowiada na pytanie lepiej niż fragment źródłowy. W pytaniu o Rządowy Fundusz Rozwoju Dróg wygrywa inna wypowiedź na ten sam temat („środki trafiają do samorządów”).

## Część II: RAG

Potok: pytanie → bi-enkoder + FAISS (50 kandydatów) → cross-enkoder (top-5) → prompt z fragmentami i metadanymi (mówca, klub, data) → LLM z API CLARIN (`/chat/completions` przez `langchain-openai`).

In [12]:
SYSTEM = (
    "Jesteś asystentem odpowiadającym na pytania o debaty w Sejmie RP. Odpowiadaj po polsku, wyłącznie na podstawie podanych fragmentów stenogramów. "
    "Po każdym stwierdzeniu podaj numer źródła w nawiasie kwadratowym, np. [2]. Jeśli fragmenty nie zawierają odpowiedzi, napisz: Brak informacji w stenogramach."
)
RAG_PROMPT = ChatPromptTemplate.from_messages([("system", SYSTEM), ("human", "Fragmenty:\n{context}\n\nPytanie: {question}")])
PLAIN_PROMPT = ChatPromptTemplate.from_messages([("human", "{question}")])
RAG_MODELS = ["bielik-pl-11b-v3.0-Instruct", "pllum-12b-chat"]
print([m.id for m in OpenAI(base_url=API, api_key=KEY).models.list()])


def llm(model, **kw):
    return ChatOpenAI(model=model, base_url=API, api_key=KEY, temperature=0, max_retries=10, **kw)


def retrieve(question, k=5):
    q = encoders[best].encode([MODELS[best][1] + question], normalize_embeddings=True)
    cand = indexes[best, "Flat"].search(q, N_CAND)[1][0]
    scores = cross_encoders[best_ce].predict([(question, passages.text.iat[i]) for i in cand])
    return passages.iloc[cand[np.argsort(-scores)[:k]]]


def context(docs):
    return "\n\n".join(f"[{n}] {r.speaker} ({r.club}, {r.date}): {r.text}" for n, r in enumerate(docs.itertuples(), 1))


def rag(question, model):
    docs = retrieve(question)
    return (RAG_PROMPT | llm(model, max_tokens=500)).invoke({"context": context(docs), "question": question}).content, docs

['agent-worker-example', 'dariah-eventstotriples', 'clarin-affect-pllum', 'llama3.3', 'llama3.1-8b', 'gemma-4-31b-it', 'gemma-4-26b-a4b-it', 'deepseek-v4-flash-0731', 'pllum-fusion', 'llama', 'pllum', 'gemma-4-26B-A4B-it-FP8-Dynamic', 'bielik-pl-11b-v3.0-Instruct', 'gpt-oss-20b', 'pllum-12b-chat', 'llama', 'pllum']


In [13]:
RAG_QUERIES = [
    "Jakie argumenty padały w Sejmie w sprawie obniżenia składki zdrowotnej dla przedsiębiorców?",
    "Jak posłowie Konfederacji oceniają unijny pakt migracyjny?",
    "Jaką pomoc zapowiadał rząd dla poszkodowanych w powodzi we wrześniu 2024 roku?",
    "Jakie stanowiska zajmowały kluby w sprawie związków partnerskich?",
    "Co posłowie mówili o podwyżkach dla nauczycieli?",
    "Kto wygrał mistrzostwa świata w piłce nożnej w 2022 roku?",
    "Jakie są opinie o Tusku w Sejmie?",
]
rag_results = []
for q in RAG_QUERIES:
    plain = (PLAIN_PROMPT | llm(RAG_MODELS[0], max_tokens=500)).invoke({"question": q}).content
    answers = {m: rag(q, m) for m in RAG_MODELS}
    docs = answers[RAG_MODELS[0]][1]
    rag_results.append({"pytanie": q, "bez RAG": plain} | {m: a for m, (a, _) in answers.items()})
    print("=" * 100, "\n###", q)
    display(docs[["speaker", "club", "date", "text"]].assign(text=docs.text.str[:200]))
    print(f"--- {RAG_MODELS[0]} bez RAG:\n{plain}")
    for m, (a, _) in answers.items():
        print(f"--- {m} + RAG:\n{a}")

### Jakie argumenty padały w Sejmie w sprawie obniżenia składki zdrowotnej dla przedsiębiorców?


,speaker,club,date,text
27426,Krzysztof Habura,KO,2024-11-27,"Panie Marszałku! Wysoki Sejmie! Dzisiejszą debatę zdominowały dwa wątki. Jeden to obniżenie składki zdrowotnej dla przedsiębiorców i dla pracowników. I tu wszyscy jesteśmy zgodni, że tę stawkę powinno"
35972,Kamil Wnuk,Polska2050-TD,2025-03-19,"Pani Marszałek! Wysoka Izbo! Drodzy Rodacy! Przyszliśmy tutaj, do Sejmu, jako Polska 2050 m.in. po to, aby obliżyć składkę zdrowotną, aby walczyć z tymi, którzy walczą z przedsiębiorcami. Prawo i Spra"
35973,Krzysztof Grabczuk,KO,2025-03-19,"Szanowna Pani Marszałek! Wysoki Sejmie! Wprowadzony przez PiS Polski Ład, który wprowadził składkę zdrowotną, niestety miał bardzo negatywny wpływ na wielu polskich przedsiębiorców, zwłaszcza tych mał"
21688,Ewa Szymanowska,Polska2050-TD,2024-10-10,"Szanowny Panie Marszałku! Państwo Ministrowie! Wysoka Izbo! Panie pośle Zandberg, na wczorajszym posiedzeniu powiedział pan, że obniżenie składki zdrowotnej to prezent dla najzamożniejszych ludzi w Po"
26495,Rafał Bochenek,PiS,2024-11-21,"że nie tylko jest źle, ale też będzie jeszcze gorzej. Rolą odpowiedzialnych rządzących jest elastyczne reagowanie na to, co się dzieje w przestrzeni społeczno-gospodarczej, i wprowadzanie instrumentów"


--- bielik-pl-11b-v3.0-Instruct bez RAG:
W Sejmie w sprawie obniżenia składki zdrowotnej dla przedsiębiorców padały różne argumenty. Zwolennicy obniżenia składki argumentowali, że obecny system jest zbyt obciążący dla małych i średnich przedsiębiorstw, co hamuje ich rozwój i innowacyjność. Wskazywali na fakt, że wysokie składki zdrowotne stanowią znaczną część kosztów prowadzenia działalności gospodarczej, co może zniechęcać do jej podejmowania. Argumentowali również, że obniżenie składki zdrowotnej może przyczynić się do zwiększenia konkurencyjności polskich firm na rynku międzynarodowym.

Przeciwnicy obniżenia składki zwracali uwagę na potencjalne negatywne skutki dla systemu opieki zdrowotnej. Wskazywali, że obniżenie składki może prowadzić do zmniejszenia wpływów do Narodowego Funduszu Zdrowia, co z kolei może skutkować obniżeniem jakości świadczonych usług medycznych. Podkreślali również, że system opieki zdrowotnej jest finansowany nie tylko ze składek zdrowotnych, ale również z 

,speaker,club,date,text
3884,Witold Tumanowicz,Konfederacja,2023-12-21,"Panie Marszałku! Wysoka Izbo! Unia Europejska przyjęła skandaliczny pakt migracyjny. Pakt obejmuje pięć rozporządzeń, w tym najistotniejsze rozporządzenie w sprawie zarządzania azylem i migracją, któr"
84754,Sławomir Zawiślak,PiS,2026-07-30,"Panie Marszałku! Wysoka Izbo! Ponad 170 tys. obywateli podpisało się pod pogradliwie lekceważonym, skrajnie krytykowanym w lewicowo-liberalnych mediach naturalnym żądaniem, aby o polskich granicach de"
84826,Maciej Duszczyk,bez klubu,2026-07-30,"procesowanie paktu migracyjnego, to są konkluzje Rady Europejskiej. Oczywiście on stara się mówić, że był tam zapis mówiący o tym, że ma być jednomyślność, ale tam dopisano, że nie dotyczy to czegoś t"
44739,Bożena Lisowska,KO,2025-07-08,"Pani Marszałek! Wysoki Sejmie! PiS i Konfederacja straszą Polaków paktem migracyjnym, dlatego przytoczę dwa główne cele tego dokumentu. Po pierwsze, ograniczenie nielegalnej migracji poprzez wzmocnien"
12938,Piotr Müller,PiS,2024-05-15,"Szanowni Państwo! Ten pakt migracyjny wejdzie w życie za kilkanaście miesięcy, natomiast konsekwencje tego będziemy widzieć przez najbliższe dekady, jeżeli chodzi o bezpieczeństwo Unii Europejskiej. T"


--- bielik-pl-11b-v3.0-Instruct bez RAG:
Posłowie Konfederacji oceniają unijny pakt migracyjny bardzo krytycznie. Uważają, że pakt ten narusza suwerenność Polski i innych krajów członkowskich, narzucając im politykę migracyjną, na którą nie mają wpływu. Konfederacja sprzeciwia się obowiązkowym kwotom relokacji migrantów, argumentując, że decyzja o przyjmowaniu uchodźców powinna należeć wyłącznie do poszczególnych państw. Ponadto, posłowie Konfederacji obawiają się, że pakt może prowadzić do zwiększenia liczby nielegalnych migrantów w Polsce, co z kolei może zwiększyć zagrożenia związane z przestępczością i terroryzmem. W ich opinii, Unia Europejska powinna skupić się na ochronie granic zewnętrznych i wspieraniu krajów, z których pochodzą migranci, zamiast narzucać politykom migracyjnym państwom członkowskim.
--- bielik-pl-11b-v3.0-Instruct + RAG:
Posłowie Konfederacji oceniają unijny pakt migracyjny bardzo krytycznie. Witold Tumanowicz (Konfederacja) określił go jako "skandaliczny" i s

,speaker,club,date,text
56509,Michał Jaros,KO,2025-11-06,Szanowny Panie Marszałku! Wysoka Izbo! Szanowna Pani Marszałek! Szanowni Państwo Parlamentarzyści! Jednym z priorytetowych obszarów rządowej pomocy poszkodowanym w wyniku powodzi we wrześniu 2024 r. b
74904,Ireneusz Zyska,PiS,2026-05-12,"czyli dróg dojazdowych, chodników, ławek i parkingów. Dlaczego rząd tak opieszale usuwa skutki powodzi z roku 2024? Powodzianie, których wnioski o wykup nieruchomości zostały odrzucone, często słyszą"
74940,Mariusz Krystian,PiS,2026-05-12,"Wysoka Izbo! Panie Ministrze! Skoro rząd sam przyznaje w uzasadnieniu projektu, że nie wszyscy poszkodowani zdążyli odbudować lub wyremontować zniszczone domy mieszkalne oraz inną infrastrukturę, to c"
30714,Włodzimierz Czarzasty,Lewica,2025-01-09,Pani marszałek Elżbieta Witek oraz pan poseł Marcin Gwóźdź w imieniu Klubu Parlamentarnego Prawo i Sprawiedliwość zadadzą pytanie w sprawie niewystarczającej pomocy ze strony rządu dla poszkodowanych
38517,Marcin Kierwiński,KO,2025-04-24,"Szanowny Panie Marszałku! Panie i Panowie Posłowie! Wysoka Izbo! To trzecia edycja ustawy powodziowej, która ma pomagać przywracać normalność i właściwe funkcjonowanie na terenach, które dotknęła powó"


... (5 outputs left out)

,speaker,club,date,text
54969,Roman Fritz,Konfederacja,2025-10-15,"Dania cały czas zwycięża. Kto tym Duńczykom tak chce nieba przychylić, że tylko oni mają produkować futra w Unii Europejskiej?"
82053,Dariusz Matecki,PiS,2026-07-03,"Wysoka Izbo! Do prezydenta Zełenskiego z polskiego Sejmu: My jako Polacy nie będziemy Ukrainie wybierać bohaterów, ale Polska nie ma obowiązku pomagać państwu, które na bohaterów wybiera ludobójców, k"
48784,Waldemar Andzel,PiS,2025-08-05,"Panie Marszałku! Wysoka Izbo! Oświadczenie dzisiejsze mam na temat zwycięstwa Polaków w Lidze Narodów. W siatkówce jesteśmy potęgą, jak chyba w żadnym innym sporcie. Złoto Polaków w Lidze Narodów, a t"
55076,Tadeusz Tomaszewski,Lewica,2025-10-15,Oświadczenie poselskie w sprawie możliwości przyznania stypendiów sportowych medalistom The World Games 2025 Szanowny Panie Marszałku! Szanowna Izbo! W dniach 7–17 sierpnia br. w chińskim mieście Chen
81462,Adam Luboński,Polska2050-TD,2026-07-01,"Pani Marszałek! Wysoka Izbo! Dziś cały świat patrzy na mundial, na mistrzostwa świata w piłce nożnej. Każdy zachwyca się pięknymi stadionami i fantastycznymi gwiazdami piłkarskimi, a tak naprawdę te p"


--- bielik-pl-11b-v3.0-Instruct bez RAG:
Mistrzostwa świata w piłce nożnej w 2022 roku wygrała reprezentacja Argentyny. W finale, który odbył się 18 grudnia 2022 roku na Lusail Iconic Stadium w Lusail, Argentyna pokonała Francję po rzutach karnych 4:2. Po 120 minutach gry był remis 3:3. Bramki dla Argentyny zdobyli: Leo Messi, Alexis Mac Allister i Julian Alvarez, a dla Francji: Kylian Mbappe (dwie) i Theo Hernandez. Był to trzeci tytuł mistrzowski dla Argentyny w historii mistrzostw świata, po zwycięstwach w 1978 i 1986 roku.
--- bielik-pl-11b-v3.0-Instruct + RAG:
Brak informacji w stenogramach. Fragmenty dotyczą innych tematów, takich jak polityka, sport i oświadczenia poselskie, ale nie zawierają informacji o wynikach mistrzostw świata w piłce nożnej w 2022 roku.
--- pllum-12b-chat + RAG:
Brak informacji w stenogramach.
### Jakie są opinie o Tusku w Sejmie?


,speaker,club,date,text
2727,Maria Koc,PiS,2023-12-12,"Szanowna Pani Marszałek! Wysoka Izbo! Panie Premierze! Większość sejmowa przedstawia pana premiera Donalda Tuska jako męża stanu, wręcz wybawiciela narodu, a pan Donald Tusk to polityk, którego wielu"
2322,Włodzimierz Czarzasty,Lewica,2023-12-11,"Panie Marszałku! Panie Prezydencie! Panie prezydencie, chciałem panu powiedzieć, i to chyba ważne, że z moich ust te słowa padną: miło pana widzieć, polski Sejm pana pozdrawia. Proszę państwa, nie będ"
2294,Mariusz Błaszczak,PiS,2023-12-11,"Panie Marszałku! Wysoki Sejmie! Zabieram dziś głos w imieniu Klubu Parlamentarnego Prawo i Sprawiedliwość — partii, która 15 października zdobyła niemal 8 mln głosów, partii, która wygrała wybory i wp"
40211,Sylwester Tułajew,PiS,2025-05-20,"Panie Marszałku! Wysoka Izbo! Wystąpienie premiera Tuska można skwitować bardzo krótko: cynizm i draństwo. Tak, w tym jest najlepszy. I to, że tak rzadko uczestniczy w posiedzeniach Sejmu, a dzisiaj p"
2296,Mariusz Błaszczak,PiS,2023-12-11,"wobec pana polityków słychać tylko pohukiwania, obelgi i groźby w naszą stronę. Wasz jedyny program to igrzyska i rządza zemsty za to, że na 8 lat zostaliście pozbawieni władzy. My wprowadziliśmy prog"


--- bielik-pl-11b-v3.0-Instruct bez RAG:
Opinie o Donaldzie Tusku w Sejmie są zróżnicowane i zależą od przynależności politycznej posłów. 

### Opinie w Platformie Obywatelskiej (PO):
- **Pozytywne**: Wielu członków PO uważa Tuska za lidera, który skutecznie reprezentuje ich interesy i wartości. Jest postrzegany jako doświadczony polityk, który ma duże wpływy na arenie międzynarodowej.
- **Neutralne**: Niektórzy członkowie PO mogą mieć różne opinie na temat jego strategii politycznej, ale generalnie cieszą się z jego przywództwa.

### Opinie w Prawie i Sprawiedliwości (PiS):
- **Negatywne**: Członkowie PiS często krytykują Tuska, oskarżając go o działanie na szkodę Polski i bycie zbyt proeuropejskim. Jest postrzegany jako przeciwnik polityczny, który nie rozumie polskich realiów.

### Opinie w innych partiach:
- **Lewica**: Opinie są mieszane. Niektórzy doceniają jego doświadczenie i umiejętności dyplomatyczne, podczas gdy inni krytykują go za brak postępowych reform.
- **Koalicja Pols

### Analiza RAG

**Bez RAG vs z RAG.** Bielik bez kontekstu pisze ogólniki, które pasowałyby do dowolnego parlamentu. O nauczycielach podaje listę w stylu „posłowie *mogą* argumentować…”. O związkach partnerskich pisze o „klubach liberalnych / centrowych / chadeckich”, bez żadnej polskiej partii. Przy powodzi wymienia typowe formy pomocy bez liczb. Z RAG odpowiedzi są konkretne i mają źródła: podwyżki 30% i 33% (Nowacka, Łoboda), 5 mld euro z Funduszu Spójności i 732 mln zł dla przedsiębiorców po powodzi, stanowiska PiS (art. 18 konstytucji), Konfederacji i Polski 2050 w sprawie związków.

**Pytanie spoza korpusu** (mundial 2022): retriever zwraca śmieci („Dania cały czas zwycięża” o futrach, siatkarska Liga Narodów). Oba modele poprawnie odpowiadają „Brak informacji w stenogramach”, a Bielik bez RAG zna odpowiedź (Argentyna). Prompt skutecznie zamyka model w korpusie, co chroni przed halucynacjami, ale odbiera mu wiedzę ogólną.

**Odpowiedź jest tak dobra jak wyszukiwanie.**
- *„Jakie są opinie o Tusku w Sejmie?”*: 4 z 5 fragmentów to PiS (w tym dwa kawałki jednego przemówienia Błaszczaka). Odpowiedź RAG jest więc w większości krytyczna: „cynizm i draństwo” (Tułajew), współpraca z Putinem, podwyższenie wieku emerytalnego. Jedyny głos pozytywny to Czarzasty, a KO nie występuje wcale. Bez RAG Bielik wymyśla symetryczny, „wyważony” obraz wszystkich partii, którego nie ma w danych.
- *„Jak posłowie Konfederacji oceniają pakt migracyjny?”*: tylko 1 z 5 fragmentów jest od posła Konfederacji, bo klub to metadana i bi-enkoder go nie widzi. Bielik dopisuje Zawiślaka do Konfederacji („PiS, ale reprezentujący Konfederację Korony Polskiej”), mieszając metadane z treścią.
- Retriever przepuszcza też szum. Do pytania o nauczycieli trafia powitanie licealistów przez marszałka Bosaka, a do pytania o związki proceduralna wypowiedź Hołowni. Oba modele to ignorują.

**Bielik vs PLLuM.** Bielik odpowiada dłużej, wymienia mówców z klubami i zwykle cytuje źródło po każdym zdaniu (czasem zbiera cytaty na końcu). PLLuM jest zwięzły, czasem za bardzo: przy powodzi podaje jeden fakt z jednego fragmentu, a przy związkach pomija numery źródeł mimo instrukcji.

**Podsumowanie.** RAG zamienia ogólnikowe odpowiedzi LLM w odpowiedzi oparte na stenogramach i weryfikowalne przez cytowania, a na pytania spoza korpusu poprawnie odmawia odpowiedzi. Najsłabszym ogniwem jest wyszukiwanie: pytania o konkretny klub lub mówcę wymagałyby filtrowania po metadanych, a pytania wartościujące dają jednostronny kontekst, który LLM wiernie powtarza.